In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os


ModuleNotFoundError: No module named 'pandas'

In [ ]:
# 1. UPLOAD GENERATED STORES (from route_generator.ipynb)
# Load the actual optimized stores instead of generating random ones
output_dir = '../data'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
df_stores = pd.read_csv(os.path.join(output_dir, 'stores.csv'))

print(f"Loaded {len(df_stores)} stores with pre-assigned routes.")

In [ ]:
# 2. GENERATE ASSETS (The Laundry Jugs)
# 2026 ESG Specs: Every jug has a 'birth weight' and material type
assets_data = []
num_of_jugs = 100000 #update to 5,000 and eventually 1,000,000 for pilot study
for i in range(num_of_jugs):
    assets_data.append([
        f"JUG_{i:05d}", 
        "HDPE_Plastic", 
        120, # grams of plastic
        "2026-01-01" # Launch date
    ])
df_assets = pd.DataFrame(assets_data, columns=['asset_id', 'material', 'weight_grams', 'initial_date'])

#print (df_assets) 

In [ ]:
# --- 3. SETUP FOR VARIABLE RETURN RATES---
simulation_start_date = datetime(2026, 1, 1)
routes_dict = df_stores.groupby('route_id')['store_id'].apply(list).to_dict()
return_rates = [0.90, 0.80, 0.70]
all_events = []


# --- POPULATION WEIGHTS (Approximate for cities) ---
# Naperville (~150k), Schaumburg (~75k), Evanston (~78k), Oak Park (~55k), Lincoln Park (~70k)
city_weights = {
    'Naperville': 0.35,
    'Schaumburg': 0.18,
    'Evanston': 0.18,
    'Lincoln Park': 0.16,
    'Oak Park': 0.13
}

# Apply weights to stores based on their city
df_stores['weight'] = df_stores['city'].map(city_weights)



In [ ]:
# --- 4. MULTI-SCENARIO LOOP ---
for rate in return_rates:
    scenario_label = f"{int(rate*100)}% Return"
    
    # Pre-calculate store probabilities for this scenario
    store_ids = df_stores['store_id'].values
    store_probs = df_stores['weight'].values / df_stores['weight'].sum()
    
    for jug_id in df_assets['asset_id']:
        # --- NEW: THE GENESIS EVENT ---
        # We record that JUG_00001 was manufactured on Jan 1st.
        # This triggers the $1.10 cost and the Carbon Debt.
        all_events.append([jug_id, 'HUB_AURORA', 'manufactured', simulation_start_date, scenario_label])
        
        current_time = simulation_start_date + timedelta(days=np.random.randint(0, 30))
        rotations = np.random.randint(1, 11)
        
        for r in range(rotations):
            route_id = np.random.choice(list(routes_dict.keys()))
            store_id = np.random.choice(routes_dict[route_id])
            
            # --- PURCHASE ---
            current_time += timedelta(days=np.random.randint(2, 14))
            all_events.append([jug_id, store_id, 'purchased', current_time, scenario_label])
            
            current_time += timedelta(days=np.random.randint(45, 91))
            
            # --- RETURN OR LEAKAGE ---
            if np.random.random() < rate:
                current_time += timedelta(days=np.random.randint(3, 15))
                all_events.append([jug_id, store_id, 'returned_to_store', current_time, scenario_label])
                
                # WASHED
                current_time += timedelta(days=3) 
                all_events.append([jug_id, 'HUB_AURORA', 'washed', current_time, scenario_label])
            else:
                # LOST
                all_events.append([jug_id, 'UNKNOWN', 'lost_to_leakage', current_time, scenario_label])
                break # Asset is dead for this scenario

df_events = pd.DataFrame(all_events, columns=['asset_id', 'location_id', 'event_type', 'timestamp', 'scenario'])

print(df_events.head())

In [ ]:
#5. Upload data to Postgres Database
from sqlalchemy import create_engine, text
from io import StringIO
import os
import pandas as pd

# --- 5A. THE FUNCTION ---
def postgres_upsert(df, table_name, engine):
    output = StringIO()
    df.to_csv(output, sep='\t', header=False, index=False)
    output.seek(0)
    
    raw_conn = engine.raw_connection()
    cursor = raw_conn.cursor()
    try:
        # FIX: We remove the extra f'"{col}"' and just use the column names
        columns = [col for col in df.columns]
        
        cursor.copy_from(output, table_name, sep='\t', null='', columns=columns)
        raw_conn.commit()
        print(f"Successfully uploaded {len(df)} rows to {table_name}.")
    except Exception as e:
        raw_conn.rollback()
        print(f"Error uploading to {table_name}: {e}")
    finally:
        cursor.close()
        raw_conn.close()

# --- 5B. SETUP ---
engine = create_engine('postgresql+psycopg2://postgres@localhost:5432/aurora_reuse_jug_db')
target_folder = 'data'

# --- 5C. SAVE CSV BACKUPS ---
# We save the file as stores.csv to stay consistent
df_assets.to_csv(os.path.join(target_folder, 'assets.csv'), index=False)
df_events.to_csv(os.path.join(target_folder, 'logistics_events.csv'), index=False)
df_stores.to_csv(os.path.join(target_folder, 'stores.csv'), index=False)

# --- 5D. REFRESH DATABASE ---
try:
    with engine.connect() as conn:
        # Instead of just Truncate, we can use Alter Table or just stay with Truncate 
        # IF you ran the SQL in Step 1 first.
        conn.execute(text("TRUNCATE TABLE logistics_events, assets, stores CASCADE;"))
        conn.commit()
        print("Database cleared. Starting upload...")

    # --- 5E. EXECUTE UPLOADS ---
    # Ensure the order matches: Stores first (primary keys), then Assets, then Events
    postgres_upsert(df_stores, 'stores', engine)
    postgres_upsert(df_assets, 'assets', engine)
    postgres_upsert(df_events, 'logistics_events', engine)
    
    print("All tables synchronized successfully!")

except Exception as e:
    print(f"Main Process Error: {e}")